In [2]:
nvidia-smi

NameError: name 'nvidia' is not defined

In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

"""
En modell, 50 seeds, full omega-range, val=q250, träning på alla övriga q.

Upplägg:
- Hela omega-intervallet används (ingen omega < q-filtrering)
- Samma target som tidigare: mean(col2, col3)
- Samma zero-padding från omega=0 upp till lägsta omega i kurvan
- Endast modellen [128, 128, 128, 128, 128, 128] används
- Validering: q = 250 MeV
- Träning: alla övriga q som hittas i datan, inklusive q = 75 MeV
- 50 oberoende träningskörningar
- De första 10 seedsen är exakt samma som modellen hade tidigare
- Early stopping på valideringskurvan q=250, precis som i första koden
- I slutet sparas:
    * run_results.csv
    * summary_by_model.csv
    * best_model_state.pt
    * best_model_metadata.json

Bästa modellen definieras som den run som har lägst validerings-MAE.
Den sparas så att den kan laddas senare utan att behöva tränas om.
"""

from __future__ import annotations

import csv
import hashlib
import json
import math
import os
import random
import re
import time
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import Dict, List, Tuple

import numpy as np
import torch
from torch import nn


# ============================================================
# 0. Device + seeds
# ============================================================
BASE_SEED = 20260413


def set_global_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


set_global_seed(BASE_SEED)

if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
else:
    DEVICE = torch.device("cpu")

print(f"Using device: {DEVICE}")


# ============================================================
# 1. Global config
# ============================================================
DATA_ROOT = Path(".")
OUTPUT_DIR = Path("output_top1_128x6_50seeds_trainall_except_val250_fullomega")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

RUNS_CSV_PATH = OUTPUT_DIR / "run_results.csv"
SUMMARY_CSV_PATH = OUTPUT_DIR / "summary_by_model.csv"
MANIFEST_PATH = OUTPUT_DIR / "manifest.json"
LOG_PATH = OUTPUT_DIR / "run_log.txt"
BEST_MODEL_PATH = OUTPUT_DIR / "best_model_state.pt"
BEST_MODEL_META_PATH = OUTPUT_DIR / "best_model_metadata.json"

OUTPUT_CURVES = ["R00", "Rt", "Rxy", "Rzz", "R0z"]
NUM_OUTPUTS = len(OUTPUT_CURVES)

VAL_Q = 250
N_REPEATS = 50

FILE_RE = re.compile(r"^CR_q(\d+)_(R00|Rt|Rxy|Rzz|R0z)_.+\.dat$", re.IGNORECASE)

# Samma 10 seeds som den gamla top1-modellen hade:
# run_seed = BASE_SEED + model_idx * 10000 + repeat_idx
# för model_idx=1 och repeat_idx=1..10
# Här fortsätter vi samma serie upp till totalt 50 seeds.
MODEL_SEEDS = [BASE_SEED + 10000 + repeat_idx for repeat_idx in range(1, N_REPEATS + 1)]

SELECTED_MODEL_CONFIG = {
    "template_name": "top1_128x6_50seeds",
    "architecture": [128, 128, 128, 128, 128, 128],
    "activation": "gelu",
    "optimizer": "adamw",
    "lr_policy": "fixed",
    "loss_name": "mae",
    "feature_set": "base+logs",
    "normalize": True,
    "unit_system": "MeV",
}

FEATURE_SETS = {
    "base": ["q", "omega"],
    "base+dist": ["q", "omega", "q_minus_omega", "omega_over_q"],
    "base+logs": ["q", "omega", "log1p_q", "log1p_omega"],
    "base+dist+logs": [
        "q",
        "omega",
        "q_minus_omega",
        "omega_over_q",
        "log1p_q",
        "log1p_omega",
    ],
}

# Fasta träningsinställningar
MAX_EPOCHS = 3000
EARLY_STOP_PATIENCE = 80
MIN_DELTA = 1e-6
BASE_LR = 1e-3
WEIGHT_DECAY = 1e-4

WEIGHTED_MAE_ALPHA = 4.0
WEIGHTED_MAE_POWER = 1.0


# ============================================================
# 2. Utilities
# ============================================================
def log(msg: str) -> None:
    ts = time.strftime("%Y-%m-%d %H:%M:%S")
    line = f"[{ts}] {msg}"
    print(line, flush=True)
    with open(LOG_PATH, "a", encoding="utf-8") as f:
        f.write(line + "\n")


def atomic_write_text(path: Path, text: str) -> None:
    tmp = path.with_suffix(path.suffix + ".tmp")
    with open(tmp, "w", encoding="utf-8") as f:
        f.write(text)
    os.replace(tmp, path)


def atomic_write_csv(path: Path, fieldnames: List[str], rows: List[dict]) -> None:
    tmp = path.with_suffix(path.suffix + ".tmp")
    with open(tmp, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        for row in rows:
            writer.writerow(row)
    os.replace(tmp, path)


def atomic_torch_save(path: Path, obj: dict) -> None:
    tmp = path.with_suffix(path.suffix + ".tmp")
    torch.save(obj, tmp)
    os.replace(tmp, path)


def sha1_dict(d: dict) -> str:
    payload = json.dumps(d, sort_keys=True, separators=(",", ":")).encode("utf-8")
    return hashlib.sha1(payload).hexdigest()


def architecture_name(layers: List[int]) -> str:
    return "-".join(str(x) for x in layers)


def count_parameters(model: nn.Module) -> int:
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


# ============================================================
# 3. File loading + curve construction
# ============================================================
def is_response_file(path: Path) -> bool:
    return FILE_RE.match(path.name) is not None


def parse_filename(path: Path) -> Tuple[int, str]:
    m = FILE_RE.match(path.name)
    if m is None:
        raise ValueError(f"Ogiltigt filnamn: {path.name}")
    q = int(m.group(1))
    curve = m.group(2)
    return q, curve


def load_single_response_file(path: Path) -> Tuple[np.ndarray, np.ndarray]:
    arr = np.loadtxt(path)

    if arr.ndim == 1:
        arr = arr.reshape(1, -1)

    if arr.shape[0] == 3 and arr.shape[1] != 3:
        arr = arr.T

    if arr.shape[1] < 3:
        raise ValueError(f"Fil {path.name} måste ha minst 3 kolumner, fick shape={arr.shape}")

    omega = arr[:, 0].astype(np.float64)
    response = np.nanmean(arr[:, 1:3], axis=1).astype(np.float64)
    return omega, response


def fill_leading_nans_with_zero(y: np.ndarray) -> np.ndarray:
    y = y.copy()
    finite = np.isfinite(y)
    if np.any(finite):
        first_finite = int(np.argmax(finite))
        if first_finite > 0:
            y[:first_finite] = 0.0
    else:
        y[:] = 0.0
    return y


def infer_zero_padding_step(omega: np.ndarray) -> float:
    diffs = np.diff(np.sort(np.unique(omega)))
    diffs = diffs[np.isfinite(diffs) & (diffs > 1e-12)]
    if len(diffs) == 0:
        return max(float(np.min(omega)), 1.0)
    return float(np.median(diffs))


@dataclass
class QCurveData:
    q_mev: int
    omega_mev: np.ndarray
    y: np.ndarray
    weights: np.ndarray
    peaks: np.ndarray
    inferred_step_mev: float


def compute_relative_curve_weights(y: np.ndarray, alpha: float, power: float) -> Tuple[np.ndarray, np.ndarray]:
    peaks = np.max(np.abs(y), axis=0)
    peaks = np.where(peaks < 1e-12, 1.0, peaks)
    rel = np.abs(y) / peaks[None, :]
    weights = 1.0 + alpha * np.power(rel, power)
    return weights.astype(np.float64), peaks.astype(np.float64)


def build_q_curve_data(data_root: Path) -> Dict[int, QCurveData]:
    files = sorted([p for p in data_root.glob("*.dat") if is_response_file(p)])
    if not files:
        raise FileNotFoundError(
            f"Hittade inga responsfiler i {data_root.resolve()}. "
            f"Förväntade namn som CR_q75_R00_NNLO_GO_450.dat"
        )

    grouped: Dict[int, Dict[str, Tuple[np.ndarray, np.ndarray]]] = {}
    for path in files:
        q, curve = parse_filename(path)
        omega, response = load_single_response_file(path)
        grouped.setdefault(q, {})[curve] = (omega, response)

    if VAL_Q not in grouped:
        raise ValueError(f"Hittade inga filer för valideringskurvan q={VAL_Q} MeV")

    q_data: Dict[int, QCurveData] = {}

    for q in sorted(grouped.keys()):
        curves = grouped[q]
        missing_curves = [c for c in OUTPUT_CURVES if c not in curves]
        if missing_curves:
            raise ValueError(f"q={q} saknar kurvor: {missing_curves}")

        omega_ref = None
        y_cols = []

        for curve_name in OUTPUT_CURVES:
            omega, y = curves[curve_name]
            y = fill_leading_nans_with_zero(y)

            if omega_ref is None:
                omega_ref = omega.copy()
            else:
                if len(omega) != len(omega_ref) or not np.allclose(omega, omega_ref, rtol=0.0, atol=1e-9):
                    raise ValueError(
                        f"Omega-grid skiljer sig mellan kurvor för q={q}. "
                        "Skriptet antar samma omega-grid för alla 5 kurvor."
                    )

            y_cols.append(y)

        omega_ref = np.asarray(omega_ref, dtype=np.float64)
        y_mat = np.stack(y_cols, axis=1)

        mask = np.isfinite(omega_ref) & np.all(np.isfinite(y_mat), axis=1)
        omega_clean = omega_ref[mask]
        y_clean = y_mat[mask]

        if len(omega_clean) == 0:
            raise ValueError(f"Inga giltiga datapunkter kvar för q={q}")

        step = infer_zero_padding_step(omega_clean)
        omega_min = float(np.min(omega_clean))

        if omega_min > 1e-12:
            omega_zeros = np.arange(0.0, omega_min, step, dtype=np.float64)
            omega_zeros = omega_zeros[omega_zeros < omega_min - 1e-12]
        else:
            omega_zeros = np.empty((0,), dtype=np.float64)

        y_zeros = np.zeros((len(omega_zeros), NUM_OUTPUTS), dtype=np.float64)

        # Hela intervallet: ingen omega<q-filtering här
        omega_aug = np.concatenate([omega_zeros, omega_clean], axis=0)
        y_aug = np.concatenate([y_zeros, y_clean], axis=0)

        weights, peaks = compute_relative_curve_weights(
            y_aug,
            alpha=WEIGHTED_MAE_ALPHA,
            power=WEIGHTED_MAE_POWER,
        )

        q_data[q] = QCurveData(
            q_mev=q,
            omega_mev=omega_aug,
            y=y_aug,
            weights=weights,
            peaks=peaks,
            inferred_step_mev=step,
        )

    return q_data


# ============================================================
# 4. Split helper
# ============================================================
def build_single_split(q_data: Dict[int, QCurveData]) -> dict:
    available_qs = sorted(q_data.keys())

    if VAL_Q not in available_qs:
        raise ValueError(f"Validerings-q={VAL_Q} saknas")

    train_qs = [q for q in available_qs if q != VAL_Q]
    if not train_qs:
        raise ValueError("Inga tränings-q återstår efter att valideringskurvan tagits bort")

    return {
        "train_qs": train_qs,
        "val_q": VAL_Q,
    }


# ============================================================
# 5. Features + data manager
# ============================================================
def convert_energy(x_mev: float, unit_system: str) -> float:
    if unit_system == "MeV":
        return float(x_mev)
    if unit_system == "GeV":
        return float(x_mev) / 1000.0
    raise ValueError(f"Okänt enhetssystem: {unit_system}")


def build_feature_vector(q_mev: float, omega_mev: float, feature_names: List[str], unit_system: str) -> List[float]:
    q = convert_energy(q_mev, unit_system)
    omega = convert_energy(omega_mev, unit_system)
    eps = 1e-12

    values = {
        "q": q,
        "omega": omega,
        "q_minus_omega": q - omega,
        "omega_over_q": 0.0 if abs(q) < eps else omega / q,
        "log1p_q": math.log1p(max(q, 0.0)),
        "log1p_omega": math.log1p(max(omega, 0.0)),
    }
    return [float(values[name]) for name in feature_names]


class SplitDataManager:
    def __init__(
        self,
        q_data: Dict[int, QCurveData],
        feature_set_name: str,
        normalize: bool,
        unit_system: str,
        device: torch.device,
    ):
        self.q_data = q_data
        self.feature_set_name = feature_set_name
        self.feature_names = FEATURE_SETS[feature_set_name]
        self.normalize = bool(normalize)
        self.unit_system = unit_system
        self.device = device

        self.x_mean = None
        self.x_std = None
        self.y_mean = None
        self.y_std = None

    def _collect_for_qs(self, q_list: List[int]) -> Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
        xs, ys, ws, q_ids = [], [], [], []
        for q in q_list:
            pack = self.q_data[q]
            for i in range(len(pack.omega_mev)):
                x = build_feature_vector(q, float(pack.omega_mev[i]), self.feature_names, self.unit_system)
                xs.append(x)
                ys.append(pack.y[i].tolist())
                ws.append(pack.weights[i].tolist())
                q_ids.append(q)

        X = np.asarray(xs, dtype=np.float32)
        Y = np.asarray(ys, dtype=np.float32)
        W = np.asarray(ws, dtype=np.float32)
        QID = np.asarray(q_ids, dtype=np.int32)
        return X, Y, W, QID

    def configure(self, train_qs: List[int], val_q: int) -> None:
        X_train_raw, Y_train_raw, W_train, Q_train = self._collect_for_qs(train_qs)
        X_val_raw, Y_val_raw, W_val, Q_val = self._collect_for_qs([val_q])

        X_train_raw = torch.tensor(X_train_raw, dtype=torch.float32, device=self.device)
        Y_train_raw = torch.tensor(Y_train_raw, dtype=torch.float32, device=self.device)
        W_train = torch.tensor(W_train, dtype=torch.float32, device=self.device)

        X_val_raw = torch.tensor(X_val_raw, dtype=torch.float32, device=self.device)
        Y_val_raw = torch.tensor(Y_val_raw, dtype=torch.float32, device=self.device)
        W_val = torch.tensor(W_val, dtype=torch.float32, device=self.device)

        if self.normalize:
            self.x_mean = X_train_raw.mean(dim=0, keepdim=True)
            self.x_std = X_train_raw.std(dim=0, keepdim=True)
            self.y_mean = Y_train_raw.mean(dim=0, keepdim=True)
            self.y_std = Y_train_raw.std(dim=0, keepdim=True)

            self.x_std = torch.where(self.x_std < 1e-12, torch.ones_like(self.x_std), self.x_std)
            self.y_std = torch.where(self.y_std < 1e-12, torch.ones_like(self.y_std), self.y_std)
        else:
            self.x_mean = torch.zeros((1, X_train_raw.shape[1]), dtype=torch.float32, device=self.device)
            self.x_std = torch.ones((1, X_train_raw.shape[1]), dtype=torch.float32, device=self.device)
            self.y_mean = torch.zeros((1, Y_train_raw.shape[1]), dtype=torch.float32, device=self.device)
            self.y_std = torch.ones((1, Y_train_raw.shape[1]), dtype=torch.float32, device=self.device)

        self.X_train = self.x_to_model_space(X_train_raw)
        print("YADDA"  + str(X_train_raw))
        self.Y_train_raw = Y_train_raw
        self.W_train = W_train
        self.Q_train = Q_train

        self.X_val = self.x_to_model_space(X_val_raw)
        self.Y_val_raw = Y_val_raw
        self.W_val = W_val
        self.Q_val = Q_val

    def x_to_model_space(self, X_raw: torch.Tensor) -> torch.Tensor:
        return (X_raw - self.x_mean) / self.x_std

    def y_from_model_space(self, Y_model: torch.Tensor) -> torch.Tensor:
        return Y_model * self.y_std + self.y_mean

    def dataset_for_single_q(self, q: int) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor, np.ndarray]:
        pack = self.q_data[q]
        X = np.asarray(
            [build_feature_vector(q, float(w), self.feature_names, self.unit_system) for w in pack.omega_mev],
            dtype=np.float32,
        )
        Y = pack.y.astype(np.float32)
        W = pack.weights.astype(np.float32)
        omega = pack.omega_mev.astype(np.float64)

        X_t = torch.tensor(X, dtype=torch.float32, device=self.device)
        Y_t = torch.tensor(Y, dtype=torch.float32, device=self.device)
        W_t = torch.tensor(W, dtype=torch.float32, device=self.device)
        X_t = self.x_to_model_space(X_t)
        return X_t, Y_t, W_t, omega


# ============================================================
# 6. Model + loss + metrics
# ============================================================
def make_activation(name: str) -> nn.Module:
    name = name.lower()
    if name == "gelu":
        return nn.GELU()
    if name == "silu":
        return nn.SiLU()
    if name == "selu":
        return nn.SELU()
    if name == "tanh":
        return nn.Tanh()
    raise ValueError(f"Okänd activation: {name}")


class MultiOutputMLP(nn.Module):
    def __init__(self, input_dim: int, hidden_layers: List[int], output_dim: int, activation: str):
        super().__init__()
        layers: List[nn.Module] = []
        prev = input_dim
        for hidden in hidden_layers:
            layers.append(nn.Linear(prev, hidden))
            layers.append(make_activation(activation))
            prev = hidden
        layers.append(nn.Linear(prev, output_dim))
        self.net = nn.Sequential(*layers)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)


def mae_loss_raw(y_pred_raw: torch.Tensor, y_true_raw: torch.Tensor) -> torch.Tensor:
    return torch.mean(torch.abs(y_pred_raw - y_true_raw))


def mse_loss_raw(y_pred_raw: torch.Tensor, y_true_raw: torch.Tensor) -> torch.Tensor:
    return torch.mean((y_pred_raw - y_true_raw) ** 2)


def weighted_mae_loss_raw(y_pred_raw: torch.Tensor, y_true_raw: torch.Tensor, weights: torch.Tensor) -> torch.Tensor:
    err = torch.abs(y_pred_raw - y_true_raw)
    per_curve = (weights * err).sum(dim=0) / (weights.sum(dim=0) + 1e-12)
    return per_curve.mean()


def objective_value(loss_name: str, y_pred_raw: torch.Tensor, y_true_raw: torch.Tensor, weights: torch.Tensor) -> torch.Tensor:
    if loss_name == "mae":
        return mae_loss_raw(y_pred_raw, y_true_raw)
    if loss_name == "mse":
        return mse_loss_raw(y_pred_raw, y_true_raw)
    if loss_name == "weighted_mae":
        return weighted_mae_loss_raw(y_pred_raw, y_true_raw, weights)
    raise ValueError(f"Okänd loss_name: {loss_name}")


def evaluate_tensor(y_true: torch.Tensor, y_pred: torch.Tensor, weights: torch.Tensor) -> dict:
    err = y_pred - y_true
    abs_err = torch.abs(err)
    sq_err = err ** 2

    mae = torch.mean(abs_err).item()
    mse = torch.mean(sq_err).item()
    per_curve_wmae = (weights * abs_err).sum(dim=0) / (weights.sum(dim=0) + 1e-12)
    wmae = per_curve_wmae.mean().item()

    per_curve_mae = torch.mean(abs_err, dim=0).detach().cpu().numpy()
    per_curve_mse = torch.mean(sq_err, dim=0).detach().cpu().numpy()
    per_curve_wmae_np = per_curve_wmae.detach().cpu().numpy()

    return {
        "mae": float(mae),
        "mse": float(mse),
        "weighted_mae": float(wmae),
        "per_curve_mae": per_curve_mae,
        "per_curve_mse": per_curve_mse,
        "per_curve_weighted_mae": per_curve_wmae_np,
        "n_points": int(y_true.shape[0]),
    }


def predict_on_dataset(model: nn.Module, dm: SplitDataManager, X: torch.Tensor) -> torch.Tensor:
    model.eval()
    with torch.no_grad():
        pred_model = model(X)
        pred_raw = dm.y_from_model_space(pred_model)
    return pred_raw


def evaluate_model_on_dataset(model: nn.Module, dm: SplitDataManager, X: torch.Tensor, Y_raw: torch.Tensor, W: torch.Tensor) -> dict:
    pred_raw = predict_on_dataset(model, dm, X)
    return evaluate_tensor(Y_raw, pred_raw, W)


# ============================================================
# 7. Optimizer
# ============================================================
def build_optimizer(model: nn.Module, optimizer_name: str, lr: float) -> torch.optim.Optimizer:
    optimizer_name = optimizer_name.lower()
    if optimizer_name == "adamw":
        return torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=WEIGHT_DECAY)
    raise ValueError(f"Okänd optimizer: {optimizer_name}")


# ============================================================
# 8. Training
# ============================================================
@dataclass
class RunConfig:
    template_name: str
    repeat_idx: int
    train_qs: List[int]
    val_q: int
    architecture: List[int]
    activation: str
    optimizer: str
    lr_policy: str
    base_lr: float
    early_stop_patience: int
    loss_name: str
    feature_set: str
    normalize: bool
    unit_system: str

    def run_id(self) -> str:
        return sha1_dict(asdict(self))[:16]


def train_one_run(dm: SplitDataManager, cfg: RunConfig) -> dict:
    input_dim = len(FEATURE_SETS[cfg.feature_set])
    model = MultiOutputMLP(
        input_dim=input_dim,
        hidden_layers=cfg.architecture,
        output_dim=NUM_OUTPUTS,
        activation=cfg.activation,
    ).to(DEVICE)

    optimizer = build_optimizer(model, cfg.optimizer, cfg.base_lr)

    n_train = dm.X_train.shape[0]
    best_state = None
    best_metrics = None
    best_epoch = -1
    best_objective = float("inf")
    epochs_without_improvement = 0
    history = []

    t0 = time.time()

    for epoch in range(1, MAX_EPOCHS + 1):
        model.train()

        perm = torch.randperm(n_train, device=DEVICE)
        xb = dm.X_train[perm]
        yb = dm.Y_train_raw[perm]
        wb = dm.W_train[perm]

        optimizer.zero_grad(set_to_none=True)
        pred_model = model(xb)
        pred_raw = dm.y_from_model_space(pred_model)
        loss = objective_value(cfg.loss_name, pred_raw, yb, wb)
        loss.backward()
        optimizer.step()

        val_metrics = evaluate_model_on_dataset(model, dm, dm.X_val, dm.Y_val_raw, dm.W_val)
        current_objective = float(val_metrics[cfg.loss_name])

        history.append(
            {
                "epoch": epoch,
                "train_objective": float(loss.item()),
                "val_mae": val_metrics["mae"],
                "val_mse": val_metrics["mse"],
                "val_weighted_mae": val_metrics["weighted_mae"],
                "lr": float(optimizer.param_groups[0]["lr"]),
            }
        )

        improved = (best_objective - current_objective) > MIN_DELTA
        if np.isfinite(current_objective) and improved:
            best_objective = current_objective
            best_epoch = epoch
            best_metrics = val_metrics
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1

        if epochs_without_improvement >= cfg.early_stop_patience:
            break

    if best_state is not None:
        model.load_state_dict(best_state)

    runtime_sec = time.time() - t0

    result = {
        "model": model,
        "best_epoch": int(best_epoch),
        "epochs_ran": int(len(history)),
        "best_objective": float(best_objective),
        "best_metrics": best_metrics,
        "runtime_sec": float(runtime_sec),
        "history": history,
        "num_params": int(count_parameters(model)),
    }
    return result


# ============================================================
# 9. Best-model checkpoint
# ============================================================
def save_best_model_checkpoint(
    path: Path,
    meta_path: Path,
    model: nn.Module,
    dm: SplitDataManager,
    cfg: RunConfig,
    seed: int,
    result: dict,
    val_metrics: dict,
    train_qs: List[int],
    val_q: int,
) -> None:
    checkpoint = {
        "state_dict": {k: v.detach().cpu().clone() for k, v in model.state_dict().items()},
        "architecture": list(cfg.architecture),
        "activation": cfg.activation,
        "feature_set": cfg.feature_set,
        "feature_names": list(dm.feature_names),
        "normalize": bool(cfg.normalize),
        "unit_system": cfg.unit_system,
        "input_dim": len(FEATURE_SETS[cfg.feature_set]),
        "output_dim": NUM_OUTPUTS,
        "x_mean": dm.x_mean.detach().cpu().clone(),
        "x_std": dm.x_std.detach().cpu().clone(),
        "y_mean": dm.y_mean.detach().cpu().clone(),
        "y_std": dm.y_std.detach().cpu().clone(),
        "best_seed": int(seed),
        "best_epoch": int(result["best_epoch"]),
        "best_val_mae": float(val_metrics["mae"]),
        "best_val_mse": float(val_metrics["mse"]),
        "best_val_weighted_mae": float(val_metrics["weighted_mae"]),
        "train_qs": list(train_qs),
        "val_q": int(val_q),
    }

    atomic_torch_save(path, checkpoint)

    metadata = {
        "best_seed": int(seed),
        "best_epoch": int(result["best_epoch"]),
        "best_val_mae": float(val_metrics["mae"]),
        "best_val_mse": float(val_metrics["mse"]),
        "best_val_weighted_mae": float(val_metrics["weighted_mae"]),
        "architecture": list(cfg.architecture),
        "activation": cfg.activation,
        "feature_set": cfg.feature_set,
        "normalize": bool(cfg.normalize),
        "unit_system": cfg.unit_system,
        "train_qs": list(train_qs),
        "val_q": int(val_q),
        "checkpoint_path": str(path.resolve()),
    }
    atomic_write_text(meta_path, json.dumps(metadata, indent=2, ensure_ascii=False))


# ============================================================
# 10. Export helpers
# ============================================================
def export_summary(rows: List[dict], path: Path) -> None:
    if not rows:
        return

    grouped = {}
    for row in rows:
        key = row["template_name"]
        grouped.setdefault(key, []).append(row)

    out_rows = []
    for key, group in grouped.items():
        val_maes = np.array([r["val_mae"] for r in group], dtype=float)
        val_mses = np.array([r["val_mse"] for r in group], dtype=float)
        best_epochs = np.array([r["best_epoch"] for r in group], dtype=float)
        runtimes = np.array([r["runtime_sec"] for r in group], dtype=float)

        exemplar = group[0]
        best_row = min(group, key=lambda r: r["val_mae"])

        out_rows.append(
            {
                "template_name": key,
                "architecture": exemplar["architecture"],
                "activation": exemplar["activation"],
                "feature_set": exemplar["feature_set"],
                "normalize": exemplar["normalize"],
                "unit_system": exemplar["unit_system"],
                "n_repeats": len(group),
                "mean_val_mae": float(val_maes.mean()),
                "std_val_mae": float(val_maes.std(ddof=0)),
                "mean_val_mse": float(val_mses.mean()),
                "std_val_mse": float(val_mses.std(ddof=0)),
                "mean_best_epoch": float(best_epochs.mean()),
                "std_best_epoch": float(best_epochs.std(ddof=0)),
                "mean_runtime_sec": float(runtimes.mean()),
                "std_runtime_sec": float(runtimes.std(ddof=0)),
                "best_seed": int(best_row["seed"]),
                "best_val_mae": float(best_row["val_mae"]),
                "best_epoch": int(best_row["best_epoch"]),
                "best_model_path": str(BEST_MODEL_PATH.resolve()),
            }
        )

    out_rows = sorted(out_rows, key=lambda r: r["mean_val_mae"])
    fieldnames = list(out_rows[0].keys())
    atomic_write_csv(path, fieldnames, out_rows)


# ============================================================
# 11. Main
# ============================================================
def main() -> None:
    q_data = build_q_curve_data(DATA_ROOT)
    split = build_single_split(q_data)

    train_qs = split["train_qs"]
    val_q = split["val_q"]

    manifest = {
        "data_root": str(DATA_ROOT.resolve()),
        "available_qs": sorted(q_data.keys()),
        "train_qs": train_qs,
        "val_q": val_q,
        "n_repeats": N_REPEATS,
        "model_seeds": MODEL_SEEDS,
        "selected_model_config": SELECTED_MODEL_CONFIG,
        "max_epochs": MAX_EPOCHS,
        "early_stop_patience": EARLY_STOP_PATIENCE,
        "base_lr": BASE_LR,
        "min_delta": MIN_DELTA,
        "weight_decay": WEIGHT_DECAY,
        "full_interval": True,
        "omega_constraint": None,
    }
    atomic_write_text(MANIFEST_PATH, json.dumps(manifest, indent=2, ensure_ascii=False))

    log("Laddade data för q-värden: " + ", ".join(str(q) for q in sorted(q_data.keys())))
    for q in sorted(q_data.keys()):
        pack = q_data[q]
        log(
            f"q={q} | n_points={len(pack.omega_mev)} | omega_min={pack.omega_mev.min():.6f} MeV | "
            f"omega_max={pack.omega_mev.max():.6f} MeV | step~{pack.inferred_step_mev:.6f} MeV"
        )

    log(f"Train qs: {train_qs}")
    log(f"Validation q: {val_q}")
    log(f"Antal seeds: {len(MODEL_SEEDS)}")
    log(f"Seeds: {MODEL_SEEDS}")

    template = SELECTED_MODEL_CONFIG

    dm = SplitDataManager(
        q_data=q_data,
        feature_set_name=template["feature_set"],
        normalize=template["normalize"],
        unit_system=template["unit_system"],
        device=DEVICE,
    )
    dm.configure(train_qs=train_qs, val_q=val_q)

    X_val_single, Y_val_single, W_val_single, _ = dm.dataset_for_single_q(val_q)

    run_rows = []

    best_global_val_mae = float("inf")
    best_global_seed = None
    best_global_epoch = None

    total_runs = len(MODEL_SEEDS)
    run_counter = 0
    t_global = time.time()

    log(
        f"Startar modell {template['template_name']} | "
        f"arch={architecture_name(template['architecture'])} | "
        f"feats={template['feature_set']}"
    )

    for repeat_idx, run_seed in enumerate(MODEL_SEEDS, start=1):
        run_counter += 1
        set_global_seed(run_seed)

        cfg = RunConfig(
            template_name=template["template_name"],
            repeat_idx=repeat_idx,
            train_qs=train_qs,
            val_q=val_q,
            architecture=list(template["architecture"]),
            activation=template["activation"],
            optimizer=template["optimizer"],
            lr_policy=template["lr_policy"],
            base_lr=BASE_LR,
            early_stop_patience=EARLY_STOP_PATIENCE,
            loss_name=template["loss_name"],
            feature_set=template["feature_set"],
            normalize=bool(template["normalize"]),
            unit_system=template["unit_system"],
        )

        result = train_one_run(dm, cfg)
        model = result["model"]

        val_metrics = evaluate_model_on_dataset(model, dm, X_val_single, Y_val_single, W_val_single)

        row = {
            "run_id": cfg.run_id(),
            "template_name": template["template_name"],
            "repeat_idx": repeat_idx,
            "architecture": architecture_name(cfg.architecture),
            "activation": cfg.activation,
            "optimizer": cfg.optimizer,
            "lr_policy": cfg.lr_policy,
            "base_lr": cfg.base_lr,
            "early_stop_patience": cfg.early_stop_patience,
            "loss_name": cfg.loss_name,
            "feature_set": cfg.feature_set,
            "normalize": cfg.normalize,
            "unit_system": cfg.unit_system,
            "seed": run_seed,
            "train_qs": ",".join(str(q) for q in train_qs),
            "val_q": val_q,
            "num_params": result["num_params"],
            "best_epoch": result["best_epoch"],
            "epochs_ran": result["epochs_ran"],
            "runtime_sec": result["runtime_sec"],
            "val_mae": val_metrics["mae"],
            "val_mse": val_metrics["mse"],
            "val_weighted_mae": val_metrics["weighted_mae"],
        }
        run_rows.append(row)

        # Spara löpande så att inget går förlorat vid avbrott
        fieldnames = list(run_rows[0].keys())
        atomic_write_csv(RUNS_CSV_PATH, fieldnames, run_rows)
        export_summary(run_rows, SUMMARY_CSV_PATH)

        if val_metrics["mae"] < best_global_val_mae:
            best_global_val_mae = float(val_metrics["mae"])
            best_global_seed = int(run_seed)
            best_global_epoch = int(result["best_epoch"])

            save_best_model_checkpoint(
                path=BEST_MODEL_PATH,
                meta_path=BEST_MODEL_META_PATH,
                model=model,
                dm=dm,
                cfg=cfg,
                seed=run_seed,
                result=result,
                val_metrics=val_metrics,
                train_qs=train_qs,
                val_q=val_q,
            )

            log(
                f"Ny bästa modell sparad | seed={best_global_seed} | "
                f"best_epoch={best_global_epoch} | val_MAE={best_global_val_mae:.6e} | "
                f"path={BEST_MODEL_PATH}"
            )

        elapsed = time.time() - t_global
        avg_time = elapsed / run_counter
        remaining = avg_time * max(total_runs - run_counter, 0)
        hh = int(remaining // 3600)
        mm = int((remaining % 3600) // 60)
        ss = int(remaining % 60)

        log(
            f"[{run_counter}/{total_runs}] completed | repeat={repeat_idx}/{N_REPEATS} | "
            f"seed={run_seed} | best_epoch={result['best_epoch']} | "
            f"val_MAE={val_metrics['mae']:.6e} | ETA~{hh:02d}:{mm:02d}:{ss:02d}"
        )

    total_elapsed = time.time() - t_global

    print(f"\nBEST SEED: {best_global_seed}")
    print(f"BEST VAL MAE: {best_global_val_mae:.6e}")
    print(f"BEST EPOCH: {best_global_epoch}")
    print(f"BEST MODEL PATH: {BEST_MODEL_PATH}\n")

    log(
        f"KLART | bästa seed={best_global_seed} | bästa val_MAE={best_global_val_mae:.6e} | "
        f"bästa epoch={best_global_epoch} | checkpoint={BEST_MODEL_PATH}"
    )
    log(
        f"total_elapsed_sec={total_elapsed:.2f} | filer: "
        f"{RUNS_CSV_PATH}, {SUMMARY_CSV_PATH}, {BEST_MODEL_PATH}, {BEST_MODEL_META_PATH}"
    )


if __name__ == "__main__":
    main()

Using device: cuda


C:\Users\local_ghasemim\Temp\ipykernel_29200\3310401276.py:213: RuntimeWarning: Mean of empty slice
  response = np.nanmean(arr[:, 1:3], axis=1).astype(np.float64)


[2026-04-16 14:10:46] Laddade data för q-värden: 50, 75, 100, 150, 200, 250, 300, 350, 400
[2026-04-16 14:10:46] q=50 | n_points=4113 | omega_min=0.000000 MeV | omega_max=513.984005 MeV | step~0.125000 MeV
[2026-04-16 14:10:46] q=75 | n_points=4114 | omega_min=0.000000 MeV | omega_max=514.089012 MeV | step~0.125000 MeV
[2026-04-16 14:10:46] q=100 | n_points=4115 | omega_min=0.000000 MeV | omega_max=514.232801 MeV | step~0.125000 MeV
[2026-04-16 14:10:46] q=150 | n_points=4119 | omega_min=0.000000 MeV | omega_max=514.656048 MeV | step~0.125000 MeV
[2026-04-16 14:10:46] q=200 | n_points=4123 | omega_min=0.000000 MeV | omega_max=515.231203 MeV | step~0.125000 MeV
[2026-04-16 14:10:46] q=250 | n_points=4130 | omega_min=0.000000 MeV | omega_max=516.000134 MeV | step~0.125000 MeV
[2026-04-16 14:10:46] q=300 | n_points=4137 | omega_min=0.000000 MeV | omega_max=516.924194 MeV | step~0.125000 MeV
[2026-04-16 14:10:46] q=350 | n_points=4146 | omega_min=0.000000 MeV | omega_max=518.016263 MeV | s

In [5]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

"""
Tre separata träningskörningar med bästa seed från tidigare sweep:
- target = min
- target = mean = (min + max) / 2
- target = max

Upplägg:
- Hela omega-intervallet används (ingen omega < q-filtrering)
- q = 250 MeV används som valideringskurva
- Alla andra q används i träning, inklusive q = 75 MeV
- Samma bästa seed från tidigare sweep återanvänds
- Samma modelltyp/hyperparametrar som bästa modellen återanvänds
- Early stopping på q = 250, precis som tidigare
- 5 figurer sparas: en per responskurva (R00, Rt, Rxy, Rzz, R0z)
  Varje figur har 3 paneler:
    * vänster: min-target
    * mitten: mean-target
    * höger: max-target
  och visar prediction vs true på q = 250 MeV
- Dessutom sparas en checkpoint per target så att modellerna kan laddas senare
"""

from __future__ import annotations

import csv
import hashlib
import json
import math
import os
import random
import re
import time
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import Dict, List, Tuple, Optional

import matplotlib.pyplot as plt
import numpy as np
import torch
from torch import nn


# ============================================================
# 0. Device + seeds
# ============================================================
BASE_SEED = 20260413


def set_global_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


set_global_seed(BASE_SEED)

if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
else:
    DEVICE = torch.device("cpu")

print(f"Using device: {DEVICE}")


# ============================================================
# 1. Global config
# ============================================================
DATA_ROOT = Path(".")
VAL_Q = 250

OUTPUT_DIR = Path("output_bestseed_min_mean_max_val250_fullomega")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

RUNS_CSV_PATH = OUTPUT_DIR / "run_results.csv"
SUMMARY_JSON_PATH = OUTPUT_DIR / "summary.json"
MANIFEST_OUT_PATH = OUTPUT_DIR / "manifest.json"
LOG_PATH = OUTPUT_DIR / "run_log.txt"
PLOTS_DIR = OUTPUT_DIR / "plots"
PLOTS_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINTS_DIR = OUTPUT_DIR / "checkpoints"
CHECKPOINTS_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_CURVES = ["R00", "Rt", "Rxy", "Rzz", "R0z"]
NUM_OUTPUTS = len(OUTPUT_CURVES)

TARGET_MODES = ["min", "mean", "max"]
TARGET_TITLES = {
    "min": "Min target",
    "mean": "Mean target",
    "max": "Max target",
}

FILE_RE = re.compile(r"^CR_q(\d+)_(R00|Rt|Rxy|Rzz|R0z)_.+\.dat$", re.IGNORECASE)

CANONICAL_CURVE_NAMES = {
    "r00": "R00",
    "rt": "Rt",
    "rxy": "Rxy",
    "rzz": "Rzz",
    "r0z": "R0z",
}

# Fallbacks om metadata/manifest inte hittas automatiskt
FALLBACK_BEST_SEED = 20270445
FALLBACK_MODEL_CONFIG = {
    "template_name": "top1_128x6_bestseed",
    "architecture": [128, 128, 128, 128, 128, 128],
    "activation": "gelu",
    "optimizer": "adamw",
    "lr_policy": "fixed",
    "loss_name": "mae",
    "feature_set": "base+logs",
    "normalize": True,
    "unit_system": "MeV",
}

FEATURE_SETS = {
    "base": ["q", "omega"],
    "base+dist": ["q", "omega", "q_minus_omega", "omega_over_q"],
    "base+logs": ["q", "omega", "log1p_q", "log1p_omega"],
    "base+dist+logs": [
        "q",
        "omega",
        "q_minus_omega",
        "omega_over_q",
        "log1p_q",
        "log1p_omega",
    ],
}

# Fasta träningsinställningar
MAX_EPOCHS = 3000
EARLY_STOP_PATIENCE = 80
MIN_DELTA = 1e-6
BASE_LR = 1e-3
WEIGHT_DECAY = 1e-4

WEIGHTED_MAE_ALPHA = 4.0
WEIGHTED_MAE_POWER = 1.0


# ============================================================
# 2. Utilities
# ============================================================
def log(msg: str) -> None:
    ts = time.strftime("%Y-%m-%d %H:%M:%S")
    line = f"[{ts}] {msg}"
    print(line, flush=True)
    with open(LOG_PATH, "a", encoding="utf-8") as f:
        f.write(line + "\n")


def atomic_write_text(path: Path, text: str) -> None:
    tmp = path.with_suffix(path.suffix + ".tmp")
    with open(tmp, "w", encoding="utf-8") as f:
        f.write(text)
    os.replace(tmp, path)


def atomic_write_csv(path: Path, fieldnames: List[str], rows: List[dict]) -> None:
    tmp = path.with_suffix(path.suffix + ".tmp")
    with open(tmp, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        for row in rows:
            writer.writerow(row)
    os.replace(tmp, path)


def atomic_torch_save(path: Path, obj: dict) -> None:
    tmp = path.with_suffix(path.suffix + ".tmp")
    torch.save(obj, tmp)
    os.replace(tmp, path)


def sha1_dict(d: dict) -> str:
    payload = json.dumps(d, sort_keys=True, separators=(",", ":")).encode("utf-8")
    return hashlib.sha1(payload).hexdigest()


def architecture_name(layers: List[int]) -> str:
    return "-".join(str(x) for x in layers)


def count_parameters(model: nn.Module) -> int:
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


def resolve_first_existing(candidates: List[Path]) -> Optional[Path]:
    for path in candidates:
        if path.exists():
            return path
    return None


def load_json_if_exists(path: Optional[Path]) -> dict:
    if path is None:
        return {}
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


# ============================================================
# 3. Läs tidigare bästa seed + config
# ============================================================
def load_previous_best_setup() -> dict:
    previous_output_dir = Path("output_top1_128x6_50seeds_trainall_except_val250_fullomega")

    metadata_path = resolve_first_existing(
        [
            previous_output_dir / "best_model_metadata.json",
            Path("best_model_metadata.json"),
        ]
    )
    manifest_path = resolve_first_existing(
        [
            previous_output_dir / "manifest.json",
            Path("manifest.json"),
        ]
    )

    metadata = load_json_if_exists(metadata_path)
    manifest = load_json_if_exists(manifest_path)

    selected_cfg = manifest.get("selected_model_config", {})

    best_seed = int(metadata.get("best_seed", FALLBACK_BEST_SEED))

    model_cfg = {
        "template_name": "bestseed_min_mean_max",
        "architecture": metadata.get("architecture", selected_cfg.get("architecture", FALLBACK_MODEL_CONFIG["architecture"])),
        "activation": metadata.get("activation", selected_cfg.get("activation", FALLBACK_MODEL_CONFIG["activation"])),
        "optimizer": selected_cfg.get("optimizer", FALLBACK_MODEL_CONFIG["optimizer"]),
        "lr_policy": selected_cfg.get("lr_policy", FALLBACK_MODEL_CONFIG["lr_policy"]),
        "loss_name": selected_cfg.get("loss_name", FALLBACK_MODEL_CONFIG["loss_name"]),
        "feature_set": metadata.get("feature_set", selected_cfg.get("feature_set", FALLBACK_MODEL_CONFIG["feature_set"])),
        "normalize": bool(metadata.get("normalize", selected_cfg.get("normalize", FALLBACK_MODEL_CONFIG["normalize"]))),
        "unit_system": metadata.get("unit_system", selected_cfg.get("unit_system", FALLBACK_MODEL_CONFIG["unit_system"])),
    }

    base_lr = float(manifest.get("base_lr", BASE_LR))
    early_stop_patience = int(manifest.get("early_stop_patience", EARLY_STOP_PATIENCE))

    return {
        "best_seed": best_seed,
        "model_cfg": model_cfg,
        "metadata_path": None if metadata_path is None else str(metadata_path.resolve()),
        "manifest_path": None if manifest_path is None else str(manifest_path.resolve()),
        "base_lr": base_lr,
        "early_stop_patience": early_stop_patience,
    }


# ============================================================
# 4. File loading + curve construction
# ============================================================
def is_response_file(path: Path) -> bool:
    return FILE_RE.match(path.name) is not None


def parse_filename(path: Path) -> Tuple[int, str]:
    m = FILE_RE.match(path.name)
    if m is None:
        raise ValueError(f"Ogiltigt filnamn: {path.name}")
    q = int(m.group(1))
    curve = CANONICAL_CURVE_NAMES[m.group(2).lower()]
    return q, curve


def load_single_response_file(path: Path) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    arr = np.loadtxt(path)

    if arr.ndim == 1:
        arr = arr.reshape(1, -1)

    if arr.shape[0] == 3 and arr.shape[1] != 3:
        arr = arr.T

    if arr.shape[1] < 3:
        raise ValueError(f"Fil {path.name} måste ha minst 3 kolumner, fick shape={arr.shape}")

    omega = arr[:, 0].astype(np.float64)
    y_min = arr[:, 1].astype(np.float64)
    y_max = arr[:, 2].astype(np.float64)
    return omega, y_min, y_max


def select_target_from_minmax(y_min: np.ndarray, y_max: np.ndarray, mode: str) -> np.ndarray:
    if mode == "min":
        return y_min.astype(np.float64)
    if mode == "mean":
        return np.nanmean(np.stack([y_min, y_max], axis=1), axis=1).astype(np.float64)
    if mode == "max":
        return y_max.astype(np.float64)
    raise ValueError(f"Okänd target mode: {mode}")


def fill_leading_nans_with_zero(y: np.ndarray) -> np.ndarray:
    y = y.copy()
    finite = np.isfinite(y)
    if np.any(finite):
        first_finite = int(np.argmax(finite))
        if first_finite > 0:
            y[:first_finite] = 0.0
    else:
        y[:] = 0.0
    return y


def infer_zero_padding_step(omega: np.ndarray) -> float:
    diffs = np.diff(np.sort(np.unique(omega)))
    diffs = diffs[np.isfinite(diffs) & (diffs > 1e-12)]
    if len(diffs) == 0:
        return max(float(np.min(omega)), 1.0)
    return float(np.median(diffs))


@dataclass
class QCurveData:
    q_mev: int
    omega_mev: np.ndarray
    y: np.ndarray
    weights: np.ndarray
    peaks: np.ndarray
    inferred_step_mev: float


def compute_relative_curve_weights(y: np.ndarray, alpha: float, power: float) -> Tuple[np.ndarray, np.ndarray]:
    peaks = np.max(np.abs(y), axis=0)
    peaks = np.where(peaks < 1e-12, 1.0, peaks)
    rel = np.abs(y) / peaks[None, :]
    weights = 1.0 + alpha * np.power(rel, power)
    return weights.astype(np.float64), peaks.astype(np.float64)


def build_q_curve_data(data_root: Path, target_mode: str) -> Dict[int, QCurveData]:
    files = sorted([p for p in data_root.glob("*.dat") if is_response_file(p)])
    if not files:
        raise FileNotFoundError(
            f"Hittade inga responsfiler i {data_root.resolve()}. "
            f"Förväntade namn som CR_q75_R00_NNLO_GO_450.dat"
        )

    grouped: Dict[int, Dict[str, Tuple[np.ndarray, np.ndarray]]] = {}
    for path in files:
        q, curve = parse_filename(path)
        omega, y_min, y_max = load_single_response_file(path)
        response = select_target_from_minmax(y_min, y_max, target_mode)
        grouped.setdefault(q, {})[curve] = (omega, response)

    if VAL_Q not in grouped:
        raise ValueError(f"Hittade inga filer för valideringskurvan q={VAL_Q} MeV")

    q_data: Dict[int, QCurveData] = {}

    for q in sorted(grouped.keys()):
        curves = grouped[q]
        missing_curves = [c for c in OUTPUT_CURVES if c not in curves]
        if missing_curves:
            raise ValueError(f"q={q} saknar kurvor: {missing_curves}")

        omega_ref = None
        y_cols = []

        for curve_name in OUTPUT_CURVES:
            omega, y = curves[curve_name]
            y = fill_leading_nans_with_zero(y)

            if omega_ref is None:
                omega_ref = omega.copy()
            else:
                if len(omega) != len(omega_ref) or not np.allclose(omega, omega_ref, rtol=0.0, atol=1e-9):
                    raise ValueError(
                        f"Omega-grid skiljer sig mellan kurvor för q={q}. "
                        "Skriptet antar samma omega-grid för alla 5 kurvor."
                    )

            y_cols.append(y)

        omega_ref = np.asarray(omega_ref, dtype=np.float64)
        y_mat = np.stack(y_cols, axis=1)

        mask = np.isfinite(omega_ref) & np.all(np.isfinite(y_mat), axis=1)
        omega_clean = omega_ref[mask]
        y_clean = y_mat[mask]

        if len(omega_clean) == 0:
            raise ValueError(f"Inga giltiga datapunkter kvar för q={q}")

        step = infer_zero_padding_step(omega_clean)
        omega_min = float(np.min(omega_clean))

        if omega_min > 1e-12:
            omega_zeros = np.arange(0.0, omega_min, step, dtype=np.float64)
            omega_zeros = omega_zeros[omega_zeros < omega_min - 1e-12]
        else:
            omega_zeros = np.empty((0,), dtype=np.float64)

        y_zeros = np.zeros((len(omega_zeros), NUM_OUTPUTS), dtype=np.float64)

        # Hela intervallet: ingen omega<q-filtering här
        omega_aug = np.concatenate([omega_zeros, omega_clean], axis=0)
        y_aug = np.concatenate([y_zeros, y_clean], axis=0)

        weights, peaks = compute_relative_curve_weights(
            y_aug,
            alpha=WEIGHTED_MAE_ALPHA,
            power=WEIGHTED_MAE_POWER,
        )

        q_data[q] = QCurveData(
            q_mev=q,
            omega_mev=omega_aug,
            y=y_aug,
            weights=weights,
            peaks=peaks,
            inferred_step_mev=step,
        )

    return q_data


# ============================================================
# 5. Split helper
# ============================================================
def build_single_split(q_data: Dict[int, QCurveData]) -> dict:
    available_qs = sorted(q_data.keys())

    if VAL_Q not in available_qs:
        raise ValueError(f"Validerings-q={VAL_Q} saknas")

    train_qs = [q for q in available_qs if q != VAL_Q]
    if not train_qs:
        raise ValueError("Inga tränings-q återstår efter att valideringskurvan tagits bort")

    return {
        "train_qs": train_qs,
        "val_q": VAL_Q,
    }


# ============================================================
# 6. Features + data manager
# ============================================================
def convert_energy(x_mev: float, unit_system: str) -> float:
    if unit_system == "MeV":
        return float(x_mev)
    if unit_system == "GeV":
        return float(x_mev) * 1000.0
    raise ValueError(f"Okänt enhetssystem: {unit_system}")


def build_feature_vector(q_mev: float, omega_mev: float, feature_names: List[str], unit_system: str) -> List[float]:
    q = convert_energy(q_mev, unit_system)
    omega = convert_energy(omega_mev, unit_system)
    eps = 1e-12

    values = {
        "q": q,
        "omega": omega,
        "q_minus_omega": q - omega,
        "omega_over_q": 0.0 if abs(q) < eps else omega / q,
        "log1p_q": math.log1p(max(q, 0.0)),
        "log1p_omega": math.log1p(max(omega, 0.0)),
    }
    return [float(values[name]) for name in feature_names]


class SplitDataManager:
    def __init__(
        self,
        q_data: Dict[int, QCurveData],
        feature_set_name: str,
        normalize: bool,
        unit_system: str,
        device: torch.device,
    ):
        self.q_data = q_data
        self.feature_set_name = feature_set_name
        self.feature_names = FEATURE_SETS[feature_set_name]
        self.normalize = bool(normalize)
        self.unit_system = unit_system
        self.device = device

        self.x_mean = None
        self.x_std = None
        self.y_mean = None
        self.y_std = None

    def _collect_for_qs(self, q_list: List[int]) -> Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
        xs, ys, ws, q_ids = [], [], [], []
        for q in q_list:
            pack = self.q_data[q]
            for i in range(len(pack.omega_mev)):
                x = build_feature_vector(q, float(pack.omega_mev[i]), self.feature_names, self.unit_system)
                xs.append(x)
                ys.append(pack.y[i].tolist())
                ws.append(pack.weights[i].tolist())
                q_ids.append(q)

        X = np.asarray(xs, dtype=np.float32)
        Y = np.asarray(ys, dtype=np.float32)
        W = np.asarray(ws, dtype=np.float32)
        QID = np.asarray(q_ids, dtype=np.int32)
        return X, Y, W, QID

    def configure(self, train_qs: List[int], val_q: int) -> None:
        X_train_raw, Y_train_raw, W_train, Q_train = self._collect_for_qs(train_qs)
        X_val_raw, Y_val_raw, W_val, Q_val = self._collect_for_qs([val_q])
        
        X_train_raw = torch.tensor(X_train_raw, dtype=torch.float32, device=self.device)
        Y_train_raw = torch.tensor(Y_train_raw, dtype=torch.float32, device=self.device)
        W_train = torch.tensor(W_train, dtype=torch.float32, device=self.device)

        X_val_raw = torch.tensor(X_val_raw, dtype=torch.float32, device=self.device)
        Y_val_raw = torch.tensor(Y_val_raw, dtype=torch.float32, device=self.device)
        W_val = torch.tensor(W_val, dtype=torch.float32, device=self.device)

        if self.normalize:
            self.x_mean = X_train_raw.mean(dim=0, keepdim=True)
            self.x_std = X_train_raw.std(dim=0, keepdim=True, unbiased=False)
            self.y_mean = Y_train_raw.mean(dim=0, keepdim=True)
            self.y_std = Y_train_raw.std(dim=0, keepdim=True, unbiased=False)

            self.x_std = torch.where(self.x_std < 1e-12, torch.ones_like(self.x_std), self.x_std)
            self.y_std = torch.where(self.y_std < 1e-12, torch.ones_like(self.y_std), self.y_std)
        else:
            self.x_mean = torch.zeros((1, X_train_raw.shape[1]), dtype=torch.float32, device=self.device)
            self.x_std = torch.ones((1, X_train_raw.shape[1]), dtype=torch.float32, device=self.device)
            self.y_mean = torch.zeros((1, Y_train_raw.shape[1]), dtype=torch.float32, device=self.device)
            self.y_std = torch.ones((1, Y_train_raw.shape[1]), dtype=torch.float32, device=self.device)

        self.X_train = self.x_to_model_space(X_train_raw)
        self.Y_train_raw = Y_train_raw
        # print(np.min(Y_train_raw), np.max(Y_train_raw))
        self.W_train = W_train
        self.Q_train = Q_train

        self.X_val = self.x_to_model_space(X_val_raw)
        self.Y_val_raw = Y_val_raw
        self.W_val = W_val
        self.Q_val = Q_val

    def x_to_model_space(self, X_raw: torch.Tensor) -> torch.Tensor:
        return (X_raw - self.x_mean) / self.x_std

    def y_from_model_space(self, Y_model: torch.Tensor) -> torch.Tensor:
        return Y_model * self.y_std + self.y_mean

    def dataset_for_single_q(self, q: int) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor, np.ndarray]:
        pack = self.q_data[q]
        X = np.asarray(
            [build_feature_vector(q, float(w), self.feature_names, self.unit_system) for w in pack.omega_mev],
            dtype=np.float32,
        )
        Y = pack.y.astype(np.float32)
        W = pack.weights.astype(np.float32)
        omega = pack.omega_mev.astype(np.float64)

        X_t = torch.tensor(X, dtype=torch.float32, device=self.device)
        Y_t = torch.tensor(Y, dtype=torch.float32, device=self.device)
        W_t = torch.tensor(W, dtype=torch.float32, device=self.device)
        X_t = self.x_to_model_space(X_t)
        return X_t, Y_t, W_t, omega


# ============================================================
# 7. Model + loss + metrics
# ============================================================
def make_activation(name: str) -> nn.Module:
    name = name.lower()
    if name == "gelu":
        return nn.GELU()
    if name == "silu":
        return nn.SiLU()
    if name == "selu":
        return nn.SELU()
    if name == "tanh":
        return nn.Tanh()
    raise ValueError(f"Okänd activation: {name}")


class MultiOutputMLP(nn.Module):
    def __init__(self, input_dim: int, hidden_layers: List[int], output_dim: int, activation: str):
        super().__init__()
        layers: List[nn.Module] = []
        prev = input_dim
        for hidden in hidden_layers:
            layers.append(nn.Linear(prev, hidden))
            layers.append(make_activation(activation))
            prev = hidden
        layers.append(nn.Linear(prev, output_dim))
        self.net = nn.Sequential(*layers)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)


def mae_loss_raw(y_pred_raw: torch.Tensor, y_true_raw: torch.Tensor) -> torch.Tensor:
    return torch.mean(torch.abs(y_pred_raw - y_true_raw))


def mse_loss_raw(y_pred_raw: torch.Tensor, y_true_raw: torch.Tensor) -> torch.Tensor:
    return torch.mean((y_pred_raw - y_true_raw) ** 2)


def weighted_mae_loss_raw(y_pred_raw: torch.Tensor, y_true_raw: torch.Tensor, weights: torch.Tensor) -> torch.Tensor:
    err = torch.abs(y_pred_raw - y_true_raw)
    per_curve = (weights * err).sum(dim=0) / (weights.sum(dim=0) + 1e-12)
    return per_curve.mean()


def objective_value(loss_name: str, y_pred_raw: torch.Tensor, y_true_raw: torch.Tensor, weights: torch.Tensor) -> torch.Tensor:
    if loss_name == "mae":
        return mae_loss_raw(y_pred_raw, y_true_raw)
    if loss_name == "mse":
        return mse_loss_raw(y_pred_raw, y_true_raw)
    if loss_name == "weighted_mae":
        return weighted_mae_loss_raw(y_pred_raw, y_true_raw, weights)
    raise ValueError(f"Okänd loss_name: {loss_name}")


def evaluate_tensor(y_true: torch.Tensor, y_pred: torch.Tensor, weights: torch.Tensor) -> dict:
    err = y_pred - y_true
    abs_err = torch.abs(err)
    sq_err = err ** 2

    mae = torch.mean(abs_err).item()
    mse = torch.mean(sq_err).item()
    per_curve_wmae = (weights * abs_err).sum(dim=0) / (weights.sum(dim=0) + 1e-12)
    wmae = per_curve_wmae.mean().item()

    per_curve_mae = torch.mean(abs_err, dim=0).detach().cpu().numpy()
    per_curve_mse = torch.mean(sq_err, dim=0).detach().cpu().numpy()
    per_curve_wmae_np = per_curve_wmae.detach().cpu().numpy()

    return {
        "mae": float(mae),
        "mse": float(mse),
        "weighted_mae": float(wmae),
        "per_curve_mae": per_curve_mae,
        "per_curve_mse": per_curve_mse,
        "per_curve_weighted_mae": per_curve_wmae_np,
        "n_points": int(y_true.shape[0]),
    }


def predict_on_dataset(model: nn.Module, dm: SplitDataManager, X: torch.Tensor) -> torch.Tensor:
    model.eval()
    with torch.no_grad():
        pred_model = model(X)
        pred_raw = dm.y_from_model_space(pred_model)
    return pred_raw


def evaluate_model_on_dataset(model: nn.Module, dm: SplitDataManager, X: torch.Tensor, Y_raw: torch.Tensor, W: torch.Tensor) -> dict:
    pred_raw = predict_on_dataset(model, dm, X)
    return evaluate_tensor(Y_raw, pred_raw, W)


# ============================================================
# 8. Optimizer
# ============================================================
def build_optimizer(model: nn.Module, optimizer_name: str, lr: float) -> torch.optim.Optimizer:
    optimizer_name = optimizer_name.lower()
    if optimizer_name == "adamw":
        return torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=WEIGHT_DECAY)
    raise ValueError(f"Okänd optimizer: {optimizer_name}")


# ============================================================
# 9. Training
# ============================================================
@dataclass
class RunConfig:
    target_mode: str
    seed: int
    train_qs: List[int]
    val_q: int
    architecture: List[int]
    activation: str
    optimizer: str
    lr_policy: str
    base_lr: float
    early_stop_patience: int
    loss_name: str
    feature_set: str
    normalize: bool
    unit_system: str

    def run_id(self) -> str:
        return sha1_dict(asdict(self))[:16]


def train_one_run(dm: SplitDataManager, cfg: RunConfig) -> dict:
    input_dim = len(FEATURE_SETS[cfg.feature_set])
    model = MultiOutputMLP(
        input_dim=input_dim,
        hidden_layers=cfg.architecture,
        output_dim=NUM_OUTPUTS,
        activation=cfg.activation,
    ).to(DEVICE)

    optimizer = build_optimizer(model, cfg.optimizer, cfg.base_lr)

    n_train = dm.X_train.shape[0]
    best_state = None
    best_metrics = None
    best_epoch = -1
    best_objective = float("inf")
    epochs_without_improvement = 0
    history = []

    t0 = time.time()

    for epoch in range(1, MAX_EPOCHS + 1):
        model.train()

        perm = torch.randperm(n_train, device=DEVICE)
        xb = dm.X_train[perm]
        yb = dm.Y_train_raw[perm]
        wb = dm.W_train[perm]

        optimizer.zero_grad(set_to_none=True)
        pred_model = model(xb)
        pred_raw = dm.y_from_model_space(pred_model)
        loss = objective_value(cfg.loss_name, pred_raw, yb, wb)
        loss.backward()
        optimizer.step()

        val_metrics = evaluate_model_on_dataset(model, dm, dm.X_val, dm.Y_val_raw, dm.W_val)
        current_objective = float(val_metrics[cfg.loss_name])

        history.append(
            {
                "epoch": epoch,
                "train_objective": float(loss.item()),
                "val_mae": val_metrics["mae"],
                "val_mse": val_metrics["mse"],
                "val_weighted_mae": val_metrics["weighted_mae"],
                "lr": float(optimizer.param_groups[0]["lr"]),
            }
        )

        improved = (best_objective - current_objective) > MIN_DELTA
        if np.isfinite(current_objective) and improved:
            best_objective = current_objective
            best_epoch = epoch
            best_metrics = val_metrics
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1

        if epochs_without_improvement >= cfg.early_stop_patience:
            break

    if best_state is None:
        raise RuntimeError(f"Ingen giltig checkpoint hittades för target_mode={cfg.target_mode}")

    model.load_state_dict(best_state)
    runtime_sec = time.time() - t0

    result = {
        "model": model,
        "best_epoch": int(best_epoch),
        "epochs_ran": int(len(history)),
        "best_objective": float(best_objective),
        "best_metrics": best_metrics,
        "runtime_sec": float(runtime_sec),
        "history": history,
        "num_params": int(count_parameters(model)),
    }
    return result

def extract_curves(model):
    model.eval()
    

    maxval = 1200
    stepsize = 10
    steps = int(maxval/stepsize)

    qcoords = np.arange(0, 1200, stepsize)
    wcoords = np.arange(0, 1200, stepsize)
    # w_coords = w_coords.flatten()
    # q_coords = q_coords.flatten()

    q_idxs, w_idxs = np.meshgrid(np.arange(0, steps), np.arange(0, steps))

    outarrays = [[0]*steps*steps for _ in range(5)]

    for (w_idx, q_idx) in zip(w_idxs.flatten(), q_idxs.flatten()):
        index = modules.nuwro.Stupid_2D_to_1D(q_idx, w_idx, steps)
        q = qcoords[q_idx]
        w = wcoords[w_idx]
        if w > q:
            continue # nonphysical
        for i in range(5):
            val = funcs[i](qcoords[q_idx], wcoords[w_idx])
            if np.isnan(val):
                # print(f"NaN at {qcoords[q_idx]}, {wcoords[w_idx]}")
                pass
            outarrays[i][index] = val

    print(wcoords)

    outarrays = np.nan_to_num(outarrays)
    

# ============================================================
# 10. Checkpoint save
# ============================================================
def save_model_checkpoint(
    path: Path,
    model: nn.Module,
    dm: SplitDataManager,
    cfg: RunConfig,
    result: dict,
    val_metrics: dict,
) -> None:
    checkpoint = {
        "state_dict": {k: v.detach().cpu().clone() for k, v in model.state_dict().items()},
        "architecture": list(cfg.architecture),
        "activation": cfg.activation,
        "feature_set": cfg.feature_set,
        "feature_names": list(dm.feature_names),
        "normalize": bool(cfg.normalize),
        "unit_system": cfg.unit_system,
        "input_dim": len(FEATURE_SETS[cfg.feature_set]),
        "output_dim": NUM_OUTPUTS,
        "x_mean": dm.x_mean.detach().cpu().clone(),
        "x_std": dm.x_std.detach().cpu().clone(),
        "y_mean": dm.y_mean.detach().cpu().clone(),
        "y_std": dm.y_std.detach().cpu().clone(),
        "seed": int(cfg.seed),
        "target_mode": cfg.target_mode,
        "best_epoch": int(result["best_epoch"]),
        "val_mae": float(val_metrics["mae"]),
        "val_mse": float(val_metrics["mse"]),
        "val_weighted_mae": float(val_metrics["weighted_mae"]),
        "train_qs": list(cfg.train_qs),
        "val_q": int(cfg.val_q),
    }
    atomic_torch_save(path, checkpoint)


# ============================================================
# 11. Plotting
# ============================================================
def plot_curve_three_targets(
    curve_name: str,
    curve_idx: int,
    store: Dict[str, dict],
    val_q: int,
    out_path: Path,
) -> None:
    fig, axes = plt.subplots(1, 3, figsize=(18, 5), sharex=False, sharey=False)

    for ax, mode in zip(axes, TARGET_MODES):
        omega = store[mode]["val_omega"]
        y_true = store[mode]["val_true"][:, curve_idx]
        y_pred = store[mode]["val_pred"][:, curve_idx]
        metrics = store[mode]["val_metrics"]

        ax.plot(omega, y_true, linewidth=2.0, label="True")
        ax.plot(omega, y_pred, linewidth=2.0, label="Prediction")

        ax.set_title(
            f"{curve_name} | {TARGET_TITLES[mode]} | q={val_q} MeV\n"
            f"MAE={metrics['per_curve_mae'][curve_idx]:.3e}"
        )
        ax.set_xlabel(r"$\omega$ [MeV]")
        ax.set_ylabel("Response")
        ax.grid(alpha=0.3)
        ax.legend(fontsize=9)

    fig.suptitle(
        f"{curve_name}: prediction vs true on validation curve q={val_q} MeV "
        f"for min / mean / max targets",
        fontsize=13,
    )
    fig.tight_layout()
    fig.savefig(out_path, dpi=180, bbox_inches="tight")
    plt.close(fig)


# ============================================================
# 12. Main
# ============================================================
def main() -> None:
    prev = load_previous_best_setup()

    BEST_SEED = int(prev["best_seed"])
    MODEL_CFG = dict(prev["model_cfg"])
    effective_base_lr = float(prev["base_lr"])
    effective_patience = int(prev["early_stop_patience"])

    manifest = {
        "data_root": str(DATA_ROOT.resolve()),
        "val_q": VAL_Q,
        "best_seed_reused": BEST_SEED,
        "previous_metadata_path": prev["metadata_path"],
        "previous_manifest_path": prev["manifest_path"],
        "model_cfg": MODEL_CFG,
        "target_modes": TARGET_MODES,
        "max_epochs": MAX_EPOCHS,
        "early_stop_patience": effective_patience,
        "base_lr": effective_base_lr,
        "min_delta": MIN_DELTA,
        "weight_decay": WEIGHT_DECAY,
        "full_interval": True,
        "omega_constraint": None,
    }
    atomic_write_text(MANIFEST_OUT_PATH, json.dumps(manifest, indent=2, ensure_ascii=False))

    log(f"Återanvänder bästa seed från tidigare körning: {BEST_SEED}")
    log(
        f"Modell: arch={architecture_name(MODEL_CFG['architecture'])} | "
        f"activation={MODEL_CFG['activation']} | "
        f"feature_set={MODEL_CFG['feature_set']} | "
        f"optimizer={MODEL_CFG['optimizer']}"
    )

    run_rows = []
    prediction_store: Dict[str, dict] = {}

    t_global = time.time()

    for target_mode in TARGET_MODES:
        log("=" * 80)
        log(f"Startar target_mode={target_mode}")

        set_global_seed(BEST_SEED)

        q_data = build_q_curve_data(DATA_ROOT, target_mode=target_mode)
        split = build_single_split(q_data)

        train_qs = split["train_qs"]
        val_q = split["val_q"]

        log("Laddade data för q-värden: " + ", ".join(str(q) for q in sorted(q_data.keys())))
        log(f"Train qs: {train_qs}")
        log(f"Validation q: {val_q}")

        dm = SplitDataManager(
            q_data=q_data,
            feature_set_name=MODEL_CFG["feature_set"],
            normalize=MODEL_CFG["normalize"],
            unit_system=MODEL_CFG["unit_system"],
            device=DEVICE,
        )
        dm.configure(train_qs=train_qs, val_q=val_q)

        X_val, Y_val, W_val, omega_val = dm.dataset_for_single_q(val_q)

        cfg = RunConfig(
            target_mode=target_mode,
            seed=BEST_SEED,
            train_qs=train_qs,
            val_q=val_q,
            architecture=list(MODEL_CFG["architecture"]),
            activation=MODEL_CFG["activation"],
            optimizer=MODEL_CFG["optimizer"],
            lr_policy=MODEL_CFG["lr_policy"],
            base_lr=effective_base_lr,
            early_stop_patience=effective_patience,
            loss_name=MODEL_CFG["loss_name"],
            feature_set=MODEL_CFG["feature_set"],
            normalize=bool(MODEL_CFG["normalize"]),
            unit_system=MODEL_CFG["unit_system"],
        )

        result = train_one_run(dm, cfg)
        print("Training finished")
        # pallar inte 2.0
        def Stupid_2D_to_1D(n: int, m: int, N: int) -> int:
            return n*N + m
        def ListToCppArray(lst: list, name: str) -> str:
            res = ",".join([str(x) for x in lst])
            return f"static double {name}[] = {{{res}}};\n"
        model = result["model"]
        model.eval()
        
        maxval = 1200
        stepsize = 4
        steps = int(maxval/stepsize)

        qcoords = np.arange(0, 1200, stepsize)
        wcoords = np.arange(0, 1200, stepsize)
        # w_coords = w_coords.flatten()
        # q_coords = q_coords.flatten()

        q_idxs, w_idxs = np.meshgrid(np.arange(0, steps), np.arange(0, steps))

        outarrays = [[0]*steps*steps for _ in range(5)]

        for (w_idx, q_idx) in zip(w_idxs.flatten(), q_idxs.flatten()):
            index = Stupid_2D_to_1D(q_idx, w_idx, steps)
            q = qcoords[q_idx]
            w = wcoords[w_idx]
            if w > q:
                continue # nonphysical
            featurevec = build_feature_vector(q, w, FEATURE_SETS[cfg.feature_set], cfg.unit_system)
            featurevec = np.asarray(featurevec, dtype=np.float32)
            featurevec = dm.x_to_model_space(torch.tensor(featurevec, device=DEVICE)).unsqueeze(0)
            with torch.no_grad():
                pred_model = model(featurevec)
                pred_raw = dm.y_from_model_space(pred_model).squeeze(0).detach().cpu().numpy()
            for i in range(5):
                val = pred_raw[0][i]
                if np.isnan(val):
                    # print(f"NaN at {qcoords[q_idx]}, {wcoords[w_idx]}")
                    pass
                outarrays[i][index] = val

        

        outarrays = np.nan_to_num(outarrays)
        outarrays [2] *= -1 # Flip Rxy, it is flipped to be easier to train on

        for (i, curvename) in enumerate(["R00", "Rxx", "Rxy", "Rzz", "R0z"]):
            res = ListToCppArray(outarrays[i], f"lfg_{curvename}")
            with open(f"data/cpp_lfg_arrays/lfg_{curvename}.h", "w") as f:
                f.write(res)

        raise Exception("Stopp o belägg")
        

        print("Extracting curves")


        val_metrics = evaluate_model_on_dataset(model, dm, X_val, Y_val, W_val)
        val_pred = predict_on_dataset(model, dm, X_val).detach().cpu().numpy()
        val_true = Y_val.detach().cpu().numpy()

        ckpt_path = CHECKPOINTS_DIR / f"bestseed_target_{target_mode}.pt"
        save_model_checkpoint(
            path=ckpt_path,
            model=model,
            dm=dm,
            cfg=cfg,
            result=result,
            val_metrics=val_metrics,
        )

        prediction_store[target_mode] = {
            "val_pred": val_pred,
            "val_true": val_true,
            "val_omega": omega_val,
            "val_metrics": val_metrics,
            "checkpoint_path": str(ckpt_path.resolve()),
        }

        row = {
            "run_id": cfg.run_id(),
            "target_mode": target_mode,
            "seed": BEST_SEED,
            "architecture": architecture_name(cfg.architecture),
            "activation": cfg.activation,
            "optimizer": cfg.optimizer,
            "lr_policy": cfg.lr_policy,
            "base_lr": cfg.base_lr,
            "early_stop_patience": cfg.early_stop_patience,
            "loss_name": cfg.loss_name,
            "feature_set": cfg.feature_set,
            "normalize": cfg.normalize,
            "unit_system": cfg.unit_system,
            "train_qs": ",".join(str(q) for q in train_qs),
            "val_q": val_q,
            "num_params": result["num_params"],
            "best_epoch": result["best_epoch"],
            "epochs_ran": result["epochs_ran"],
            "runtime_sec": result["runtime_sec"],
            "val_mae": val_metrics["mae"],
            "val_mse": val_metrics["mse"],
            "val_weighted_mae": val_metrics["weighted_mae"],
            "checkpoint_path": str(ckpt_path.resolve()),
        }
        run_rows.append(row)

        atomic_write_csv(RUNS_CSV_PATH, list(run_rows[0].keys()), run_rows)

        log(
            f"Klart target_mode={target_mode} | seed={BEST_SEED} | "
            f"best_epoch={result['best_epoch']} | "
            f"val_MAE={val_metrics['mae']:.6e} | "
            f"checkpoint={ckpt_path.name}"
        )

    # Spara 5 figurer, en per responskurva
    for curve_idx, curve_name in enumerate(OUTPUT_CURVES):
        out_path = PLOTS_DIR / f"{curve_name}_q{VAL_Q}_min_mean_max_prediction_vs_true.png"
        plot_curve_three_targets(
            curve_name=curve_name,
            curve_idx=curve_idx,
            store=prediction_store,
            val_q=VAL_Q,
            out_path=out_path,
        )
        log(f"Sparade figur: {out_path}")

    summary = {
        "best_seed_reused": BEST_SEED,
        "val_q": VAL_Q,
        "target_modes": TARGET_MODES,
        "results": run_rows,
        "plots_dir": str(PLOTS_DIR.resolve()),
    }
    atomic_write_text(SUMMARY_JSON_PATH, json.dumps(summary, indent=2, ensure_ascii=False))

    total_elapsed = time.time() - t_global
    log(f"KLART | total_elapsed_sec={total_elapsed:.2f}")
    log(f"CSV: {RUNS_CSV_PATH}")
    log(f"Summary: {SUMMARY_JSON_PATH}")
    log(f"Plots: {PLOTS_DIR}")
    log(f"Checkpoints: {CHECKPOINTS_DIR}")

    print("\nSammanfattning:")
    for row in run_rows:
        print(
            f"target={row['target_mode']:>4s} | "
            f"seed={row['seed']} | "
            f"best_epoch={row['best_epoch']} | "
            f"val_MAE={row['val_mae']:.6e} | "
            f"checkpoint={row['checkpoint_path']}"
        )
    print("Plots saved in:", PLOTS_DIR.resolve())


if __name__ == "__main__":
    main()

Using device: cuda
[2026-04-19 14:34:33] Återanvänder bästa seed från tidigare körning: 20270445
[2026-04-19 14:34:33] Modell: arch=128-128-128-128-128-128 | activation=gelu | feature_set=base+logs | optimizer=adamw
[2026-04-19 14:34:33] ================================================================================
[2026-04-19 14:34:33] Startar target_mode=min
[2026-04-19 14:34:33] Laddade data för q-värden: 50, 75, 100, 150, 200, 250, 300, 350, 400
[2026-04-19 14:34:33] Train qs: [50, 75, 100, 150, 200, 300, 350, 400]
[2026-04-19 14:34:33] Validation q: 250
Training finished


Exception: Stopp o belägg